### Volatility Targets

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Locate the processed stock data directory
possible_paths = [
    Path("notebooks/data/processed/stocks"),
    Path("data/processed/stocks"),
]

stock_dir = next(
    (path for path in possible_paths if path.exists()),
    None
)

if stock_dir is None:
    raise FileNotFoundError("Could not locate the processed stocks directory.")

# List available datasets
stock_files = sorted(stock_dir.glob("*.parquet"))

print(f"Processed stock directory: {stock_dir}")
print(f"Number of stock datasets: {len(stock_files)}")
print("\nFirst five files:")
for file in stock_files[:5]:
    print(file.name)

# inspecting one dataset
sample = pd.read_parquet(stock_files[0])

print("\nDataset shape:")
print(sample.shape)

print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nFirst five rows:")
display(sample.head())

Processed stock directory: data/processed/stocks
Number of stock datasets: 50

First five files:
ADANIENT_NS.parquet
ADANIPORTS_NS.parquet
APOLLOHOSP_NS.parquet
ASIANPAINT_NS.parquet
AXISBANK_NS.parquet

Dataset shape:
(4159, 8)

Columns:
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']

Data types:
Open            float64
High            float64
Low             float64
Close           float64
Adj Close       float64
Volume            int64
Dividends       float64
Stock Splits    float64
dtype: object

First five rows:


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2010-01-04,67.011856,69.586876,66.368103,69.219017,57.527908,1860224,0.0,0.0
2010-01-05,69.433601,71.111961,69.433601,70.820740,58.859104,2280286,0.0,0.0
2010-01-06,70.966354,79.243202,70.506531,77.810081,64.667953,7326390,0.0,0.0
2010-01-07,79.611069,79.611069,74.031853,74.598969,61.999172,4759208,0.0,0.0
2010-01-08,75.074120,77.863731,73.909233,75.541611,62.782619,3324076,0.0,0.0


In [3]:
# Locate TMPV dataset
stock_dir = Path("data/processed/stocks")
tmpv_files = list(stock_dir.glob("TMPV*.parquet"))

if len(tmpv_files) != 1:
    raise FileNotFoundError(
        f"Expected one TMPV file, found: {[f.name for f in tmpv_files]}"
    )

tmpv = pd.read_parquet(tmpv_files[0]).copy()

# Ensure dates are sorted
tmpv.index = pd.to_datetime(tmpv.index)
tmpv = tmpv.sort_index()

# Inspect the period around the demerger
window = tmpv.loc["2025-10-07":"2025-10-21"].copy()

# Calculate close-to-close log returns
window["Close_Return_%"] = (
    np.log(window["Close"] / window["Close"].shift(1)) * 100
)

window["Adj_Close_Return_%"] = (
    np.log(window["Adj Close"] / window["Adj Close"].shift(1)) * 100
)

# Compare adjusted and unadjusted prices
window["Adj_Close_Ratio"] = (
    window["Adj Close"] / window["Close"]
)

display(
    window[
        [
            "Close",
            "Adj Close",
            "Dividends",
            "Stock Splits",
            "Close_Return_%",
            "Adj_Close_Return_%",
            "Adj_Close_Ratio",
        ]
    ].round(4)
)

,Close,Adj Close,Dividends,Stock Splits,Close_Return_%,Adj_Close_Return_%,Adj_Close_Ratio
Date,,,,,,,
2025-10-07,698.05,692.3071,0.0,0.0,NaN,NaN,0.9918
2025-10-08,681.55,675.9429,0.0,0.0,-2.3921,-2.3921,0.9918
2025-10-09,681.10,675.4965,0.0,0.0,-0.0660,-0.0661,0.9918
2025-10-10,678.95,673.3643,0.0,0.0,-0.3162,-0.3162,0.9918
2025-10-13,660.75,655.3140,0.0,0.0,-2.7172,-2.7172,0.9918
2025-10-14,395.45,392.1966,0.0,0.0,-51.3351,-51.3351,0.9918
2025-10-15,390.85,387.6345,0.0,0.0,-1.1701,-1.1701,0.9918
2025-10-16,396.80,393.5355,0.0,0.0,1.5108,1.5108,0.9918
2025-10-17,396.60,393.3372,0.0,0.0,-0.0504,-0.0504,0.9918


#### Calculate Returns and Forward Volatility Targets for all 50 Stocks
- Daily log returns:

rt = ln(Pt / Pt-1)

- Forward realized volatility proxy: to estimate how much a stock's returns fluctuate over a future period, starting after the prediction date



In [4]:
stock_dir = Path("data/processed/stocks")
target_dir = Path("data/processed/volatility_targets")

target_dir.mkdir(parents=True, exist_ok=True)

HORIZONS = [5, 21]

# Known TMPV demerger-related return
TMPV_INVALID_RETURN_DATE = pd.Timestamp("2025-10-14")

def calculate_volatility_targets(df, ticker):
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Basic data checks
    if df.index.has_duplicates:
        raise ValueError(f"{ticker}: Duplicate dates found.")

    if "Adj Close" not in df.columns:
        raise ValueError(f"{ticker}: Adj Close column is missing.")

    prices = df["Adj Close"]

    if (prices.dropna() <= 0).any():
        raise ValueError(f"{ticker}: Non-positive adjusted prices found.")

    # Daily close-to-close log returns
    returns = np.log(prices / prices.shift(1))

    # Exclude the known TMPV demerger-related return
    if ticker.startswith("TMPV"):
        if TMPV_INVALID_RETURN_DATE in returns.index:
            returns.loc[TMPV_INVALID_RETURN_DATE] = np.nan

    df["Daily_Return"] = returns

    # Squared returns form the volatility proxy
    squared_returns = returns.pow(2)

    for h in HORIZONS:

        # Sum the next h squared returns, excluding today's return
        future_squared_sum = (
            squared_returns.iloc[::-1]
            .rolling(window=h, min_periods=h)
            .sum()
            .shift(1)
            .iloc[::-1]
        )

        # Annualized forward realized-volatility proxy
        df[f"RV_{h}D"] = np.sqrt(
            (252 / h) * future_squared_sum
        )

    return df[
        ["Daily_Return", "RV_5D", "RV_21D"]
    ]



stock_files = sorted(stock_dir.glob("*.parquet"))

if len(stock_files) != 50:
    raise ValueError(
        f"Expected 50 stock files, found {len(stock_files)}."
    )

for file in stock_files:

    ticker = file.stem
    stock_data = pd.read_parquet(file)

    targets = calculate_volatility_targets(
        stock_data,
        ticker
    )

    output_path = target_dir / f"{ticker}_targets.parquet"
    targets.to_parquet(output_path)

print(f"Processed {len(stock_files)} stocks.")
print(f"Target files saved to: {target_dir}")

Processed 50 stocks.
Target files saved to: data/processed/volatility_targets


In [12]:
# Inspect one stock's targets
sample_file = sorted(target_dir.glob("*.parquet"))[0]

sample = pd.read_parquet(sample_file)

display(sample.head(10))
display(sample.tail(10))

# Check target availability
print("\nMissing values:")
display(sample.isna().sum())

print("\nTarget summary:")
display(sample[["RV_5D", "RV_21D"]].describe())


,Daily_Return,RV_5D,RV_21D
Date,,,
2010-01-04,NaN,0.765567,0.522710
2010-01-05,0.022876,0.890553,0.519890
2010-01-06,0.094119,0.594152,0.413778
2010-01-07,-0.042145,0.587979,0.389027
2010-01-08,0.012557,0.581184,0.389658
2010-01-11,0.017698,0.573791,0.385179
2010-01-12,-0.068046,0.310922,0.306408
2010-01-13,-0.011266,0.304700,0.304957
2010-01-14,0.040391,0.188332,0.277383


,Daily_Return,RV_5D,RV_21D
Date,,,
2026-10-26,0.012437,0.383523,NaN
2026-10-27,-0.007203,0.384843,NaN
2026-10-28,-0.004105,0.493560,NaN
2026-10-29,0.050367,0.345971,NaN
2026-10-30,-0.010988,0.338614,NaN
2026-11-02,0.013860,NaN,NaN
2026-11-03,-0.008486,NaN,NaN
2026-11-04,0.043722,NaN,NaN
2026-11-05,0.008856,NaN,NaN



Missing values:


Daily_Return     1
RV_5D            5
RV_21D          21
dtype: int64


Target summary:


,RV_5D,RV_21D
count,4154.000000,4138.000000
mean,0.408877,0.452001
std,0.305754,0.237854
min,0.013696,0.059060
25%,0.228408,0.310093
50%,0.344652,0.402776
75%,0.495451,0.515185
max,3.654108,2.042363


# Interpretation:
Daily_Return = 0.022876 means a 2.29% daily return.


RV_5D = 0.765567 means approximately 76.56% annualized volatility.

RV_21D = 0.452001 means approximately 45.20% annualized volatility for that observation.

In [14]:
# Validate volatility targets across all stocks

validation_results = []

for file in sorted(target_dir.glob("*_targets.parquet")):

    ticker = file.name.replace("_targets.parquet", "")
    targets = pd.read_parquet(file)
    original = pd.read_parquet(stock_dir / f"{ticker}.parquet")

    validation_results.append({
        "Ticker": ticker,
        "Dates Match": targets.index.equals(original.index),
        "RV_5D Boundary OK": targets["RV_5D"].tail(5).isna().all(),
        "RV_21D Boundary OK": targets["RV_21D"].tail(21).isna().all()
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Stocks processed:", len(validation_df))
print("All checks passed:", validation_df.iloc[:, 1:].all().all())

# Verify TMPV demerger treatment
tmpv = pd.read_parquet(target_dir / "TMPV_NS_targets.parquet")

print(
    "TMPV demerger return excluded:",
    pd.isna(tmpv.loc["2025-10-14", "Daily_Return"])
)

,Ticker,Dates Match,RV_5D Boundary OK,RV_21D Boundary OK
0,ADANIENT_NS,True,True,True
1,ADANIPORTS_NS,True,True,True
2,APOLLOHOSP_NS,True,True,True
3,ASIANPAINT_NS,True,True,True
4,AXISBANK_NS,True,True,True
5,BAJAJ-AUTO_NS,True,True,True
6,BAJAJFINSV_NS,True,True,True
7,BAJFINANCE_NS,True,True,True
8,BEL_NS,True,True,True
9,BHARTIARTL_NS,True,True,True


Stocks processed: 50
All checks passed: True
TMPV demerger return excluded: True


### Lagged Returns and Historical Volatility

In [15]:
# Creating Lagged Return and Historical Volatility Features

feature_dir = Path("data/processed/features")
feature_dir.mkdir(parents=True, exist_ok=True)

return_lags = [1,2,5,10,21]
vol_windows = [5, 21, 63]

target_files = sorted(target_dir.glob("*_targets.parquet"))

if len(target_files) != 50:
    raise ValueError(f"Expected 50 target files, found {len(target_files)}")

for file in target_files:
    ticker = file.name.replace("_targets.parquet", "")

    # already validated daily returns
    targets = pd.read_parquet(file)
    returns = targets["Daily_Return"]

    features = pd.DataFrame(index=targets.index)

    # lagged daily returns
    for lag in return_lags:
        features[f"Return_Lag_{lag}"] = returns.shift(lag)

    # historical annualized volatility
    for window in vol_windows:
        historical_returns = returns.shift(1)

        features[f"Hist_Vol_{window}D"] = (
            historical_returns
            .rolling(window=window, min_periods=window)
            .std()
            * np.sqrt(252)
        )

    # save features separately
    features.to_parquet(feature_dir / f"{ticker}_features.parquet")

print(f"Created features for {len(target_files)} stocks.")
print(f"Saved to: {feature_dir}")


Created features for 50 stocks.
Saved to: data/processed/features


In [20]:
# Inspect one stock's engineered features

sample_file = sorted(feature_dir.glob("*_features.parquet"))[0]
sample_features = pd.read_parquet(sample_file)

display(sample_features.head(10).round(4))

print("Number of stocks:", len(list(feature_dir.glob("*_features.parquet"))))
print("Number of features:", sample_features.shape[1])
print("Missing values:")
display(sample_features.isna().sum())


,Return_Lag_1,Return_Lag_2,Return_Lag_5,Return_Lag_10,Return_Lag_21,Hist_Vol_5D,Hist_Vol_21D,Hist_Vol_63D
Date,,,,,,,,
2010-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,0.0229,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,0.0941,0.0229,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,-0.0421,0.0941,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-11,0.0126,-0.0421,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-12,0.0177,0.0126,0.0229,NaN,NaN,0.7703,NaN,NaN
2010-01-13,-0.0680,0.0177,0.0941,NaN,NaN,0.9944,NaN,NaN
2010-01-14,-0.0113,-0.0680,-0.0421,NaN,NaN,0.5801,NaN,NaN


Number of stocks: 50
Number of features: 8
Missing values:


Return_Lag_1      2
Return_Lag_2      3
Return_Lag_5      6
Return_Lag_10    11
Return_Lag_21    22
Hist_Vol_5D       6
Hist_Vol_21D     22
Hist_Vol_63D     64
dtype: int64

In [22]:
# Verify when each feature first becomes available

display(
    sample_features.apply(lambda col: col.first_valid_index())
)

Return_Lag_1    2010-01-06
Return_Lag_2    2010-01-07
Return_Lag_5    2010-01-12
Return_Lag_10   2010-01-19
Return_Lag_21   2010-02-04
Hist_Vol_5D     2010-01-12
Hist_Vol_21D    2010-02-04
Hist_Vol_63D    2010-04-09
dtype: datetime64[ms]

### Volume-based features


In [24]:
# Create Volume Based Features

for file in sorted(target_dir.glob("*_targets.parquet")):

    ticker = file.name.replace("_targets.parquet", "")

    stock = pd.read_parquet(stock_dir / f"{ticker}.parquet")
    features = pd.read_parquet(feature_dir / f"{ticker}_features.parquet")

    volume = stock["Volume"]

    # Daily percentage change in volume
    features["Volume_Change_1D"] = volume.pct_change()

    # Current volume relative to previous average volume
    for window in [5, 21]:
        previous_avg = volume.shift(1).rolling(
            window=window,
            min_periods=window
        ).mean()

        features[f"Volume_MA_Ratio_{window}D"] = (
            volume / previous_avg
        )

    features.to_parquet(feature_dir / f"{ticker}_features.parquet")

In [25]:
display(
    features[
        [
            "Volume_Change_1D",
            "Volume_MA_Ratio_5D",
            "Volume_MA_Ratio_21D"
        ]
    ].head(30).round(3)
)

,Volume_Change_1D,Volume_MA_Ratio_5D,Volume_MA_Ratio_21D
Date,,,
2010-01-04,NaN,NaN,NaN
2010-01-05,0.237,NaN,NaN
2010-01-06,-0.144,NaN,NaN
2010-01-07,-0.458,NaN,NaN
2010-01-08,0.809,NaN,NaN
2010-01-11,-0.283,0.758,NaN
2010-01-12,-0.522,0.382,NaN
2010-01-13,1.114,0.997,NaN
2010-01-14,1.698,2.928,NaN


### Price Range Features

In [27]:
for file in sorted(target_dir.glob("*_targets.parquet")):

    ticker = file.name.replace("_targets.parquet", "")

    stock = pd.read_parquet(stock_dir / f"{ticker}.parquet")
    features = pd.read_parquet(feature_dir / f"{ticker}_features.parquet")

    previous_close = stock["Close"].shift(1)

    features["Daily_Range"] = (
        (stock["High"] - stock["Low"]) / previous_close
    )

    features.to_parquet(feature_dir / f"{ticker}_features.parquet")

print("Price-range feature added for all 50 stocks.")


Price-range feature added for all 50 stocks.


In [28]:
display(
    features[["Daily_Range"]].head(10).mul(100).round(2)
    .rename(columns={"Daily_Range": "Daily_Range (%)"})
)

,Daily_Range (%)
Date,
2010-01-04,NaN
2010-01-05,2.07
2010-01-06,2.74
2010-01-07,2.72
2010-01-08,1.43
2010-01-11,1.81
2010-01-12,3.22
2010-01-13,6.70
2010-01-14,1.94


In [29]:
# inspecting existing processed macro datasets
macro_dir = Path("data/processed")

for file in sorted(macro_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    print(f"\n{file.name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Date range:", df.index.min(), "to", df.index.max())
    display(df.head(3))


market_macro_aligned.parquet
Shape: (4138, 9)
Columns: ['NIFTY_50_Close', 'INDIA_VIX_Close', 'INDIA_VIX_Source_Date', 'USD_INR_Close', 'USD_INR_Source_Date', 'BRENT_CRUDE_Close', 'BRENT_CRUDE_Source_Date', 'US_10Y_YIELD_Close', 'US_10Y_YIELD_Source_Date']
Date range: 2010-01-04 00:00:00 to 2026-11-06 00:00:00


,NIFTY_50_Close,INDIA_VIX_Close,INDIA_VIX_Source_Date,USD_INR_Close,USD_INR_Source_Date,BRENT_CRUDE_Close,BRENT_CRUDE_Source_Date,US_10Y_YIELD_Close,US_10Y_YIELD_Source_Date
Date,,,,,,,,,
2010-01-04,5232.200195,23.639999,2010-01-04,NaN,NaT,NaN,NaT,NaN,NaT
2010-01-05,5277.899902,22.270000,2010-01-05,46.287998,2010-01-04,80.120003,2010-01-04,3.841,2010-01-04
2010-01-06,5281.799805,22.120001,2010-01-06,46.119999,2010-01-05,80.589996,2010-01-05,3.755,2010-01-05


### Market Level and Macroeconomic Features

In [30]:
macro = pd.read_parquet(
    Path("data/processed/market_macro_aligned.parquet")
).sort_index()

# confirm unique, chronological dates
if macro.index.has_duplicates:
    raise ValueError("Duplicate dates found in macro dataset.")

# confirm no macro observation comes from after its forecast date
source_cols = [
    "INDIA_VIX_Source_Date",
    "USD_INR_Source_Date",
    "BRENT_CRUDE_Source_Date",
    "US_10Y_YIELD_Source_Date"
]

for col in source_cols:
    source_dates = pd.to_datetime(macro[col])
    if (source_dates.dropna() > macro.index[source_dates.notna()]).any():
        raise ValueError(f"Future information detected in {col}.")

# a separate market/macro feature dataset
market_features = pd.DataFrame(index=macro.index)

# NIFTY 50 market features
nifty_returns = np.log(
    macro["NIFTY_50_Close"] /
    macro["NIFTY_50_Close"].shift(1)
)

market_features["NIFTY_Return"] = nifty_returns

market_features["NIFTY_Hist_Vol_21D"] = (
    nifty_returns.rolling(21, min_periods=21).std()
    * np.sqrt(252)
)

# India VIX features
market_features["INDIA_VIX_Level"] = macro["INDIA_VIX_Close"]
market_features["INDIA_VIX_Change"] = macro["INDIA_VIX_Close"].pct_change()

# macroeconomic features
market_features["USD_INR_Return"] = np.log(
    macro["USD_INR_Close"] /
    macro["USD_INR_Close"].shift(1)
)

market_features["BRENT_CRUDE_Return"] = np.log(
    macro["BRENT_CRUDE_Close"] /
    macro["BRENT_CRUDE_Close"].shift(1)
)

market_features["US_10Y_YIELD_Change"] = (
    macro["US_10Y_YIELD_Close"].diff()
)

# saving separately from individual stock features
market_features.to_parquet(
    Path("data/processed/features/market_macro_features.parquet")
)

print("Market and macro features created:", market_features.shape[1])
print("Date range:", market_features.index.min(), "to", market_features.index.max())

display(market_features.head(10).round(4))
display(market_features.isna().sum())


Market and macro features created: 7
Date range: 2010-01-04 00:00:00 to 2026-11-06 00:00:00


,NIFTY_Return,NIFTY_Hist_Vol_21D,INDIA_VIX_Level,INDIA_VIX_Change,USD_INR_Return,BRENT_CRUDE_Return,US_10Y_YIELD_Change
Date,,,,,,,
2010-01-04,NaN,NaN,23.64,NaN,NaN,NaN,NaN
2010-01-05,0.0087,NaN,22.27,-0.0580,NaN,NaN,NaN
2010-01-06,0.0007,NaN,22.12,-0.0067,-0.0036,0.0058,-0.086
2010-01-07,-0.0035,NaN,22.50,0.0172,-0.0087,0.0160,0.053
2010-01-08,-0.0035,NaN,22.57,0.0031,-0.0007,-0.0047,0.014
2010-01-11,0.0009,NaN,22.68,0.0049,-0.0037,-0.0017,-0.014
2010-01-12,-0.0075,NaN,23.08,0.0176,-0.0035,-0.0049,0.010
2010-01-13,0.0045,NaN,21.99,-0.0472,0.0057,-0.0208,-0.099
2010-01-14,0.0049,NaN,20.98,-0.0459,-0.0029,-0.0126,0.064


NIFTY_Return            1
NIFTY_Hist_Vol_21D     21
INDIA_VIX_Level         0
INDIA_VIX_Change        1
USD_INR_Return          2
BRENT_CRUDE_Return      2
US_10Y_YIELD_Change     2
dtype: int64

### Leakage Prevention

In [33]:
target_dir = Path("data/processed/volatility_targets")
feature_dir = Path("data/processed/features")
macro_path = Path("data/processed/market_macro_aligned.parquet")

audit_results = []

for file in sorted(target_dir.glob("*_targets.parquet")):
    ticker = file.stem.replace("_targets", "")

    targets = pd.read_parquet(file)
    features = pd.read_parquet(feature_dir / f"{ticker}_features.parquet")
    stock = pd.read_parquet(stock_dir / f"{ticker}.parquet")

    # All datasets must refer to the same prediction dates
    dates_aligned = (
        targets.index.equals(features.index)
        and targets.index.equals(stock.index)
    )

    # Target columns must never appear among model predictors
    no_target_leakage = not any(
        col in features.columns
        for col in ["RV_5D", "RV_21D", "Daily_Return"]
    )

    # Feature dates must be chronological
    chronological = features.index.is_monotonic_increasing

    audit_results.append({
        "Ticker": ticker,
        "Dates Aligned": dates_aligned,
        "No Target Leakage": no_target_leakage,
        "Chronological": chronological
    })

audit = pd.DataFrame(audit_results)

display(audit.head())

,Ticker,Dates Aligned,No Target Leakage,Chronological
0,ADANIENT_NS,True,True,True
1,ADANIPORTS_NS,True,True,True
2,APOLLOHOSP_NS,True,True,True
3,ASIANPAINT_NS,True,True,True
4,AXISBANK_NS,True,True,True


### Validation Strategy

In [36]:
# Inspecting Usable target date coverage

target_dir = Path("data/processed/volatility_targets")

coverage = []

for file in sorted(target_dir.glob("*_targets.parquet")):
    ticker = file.stem.replace("_targets", "")
    targets = pd.read_parquet(file)

    valid_5d = targets["RV_5D"].dropna()
    valid_21d = targets["RV_21D"].dropna()

    coverage.append({
        "Ticker": ticker,
        "5D_Start": valid_5d.index.min(),
        "5D_End": valid_5d.index.max(),
        "21D_Start": valid_21d.index.min(),
        "21D_End": valid_21d.index.max(),
        "5D_Observations": len(valid_5d),
        "21D_Observations": len(valid_21d)
    })

coverage = pd.DataFrame(coverage)

display(coverage.head(10))

print("\nOverall usable date coverage:")
print("5D:", coverage["5D_Start"].min(), "to", coverage["5D_End"].max())
print("21D:", coverage["21D_Start"].min(), "to", coverage["21D_End"].max())

,Ticker,5D_Start,5D_End,21D_Start,21D_End,5D_Observations,21D_Observations
0,ADANIENT_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
1,ADANIPORTS_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
2,APOLLOHOSP_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
3,ASIANPAINT_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
4,AXISBANK_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
5,BAJAJ-AUTO_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
6,BAJAJFINSV_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
7,BAJFINANCE_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
8,BEL_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138
9,BHARTIARTL_NS,2010-01-04,2026-10-30,2010-01-04,2026-10-07,4154,4138



Overall usable date coverage:
5D: 2010-01-04 00:00:00 to 2026-10-30 00:00:00
21D: 2010-01-04 00:00:00 to 2026-10-07 00:00:00


In [38]:
market_features = pd.read_parquet(
    feature_dir / "market_macro_features.parquet"
)

reference_targets = pd.read_parquet(
    target_dir / "ADANIENT_NS_targets.parquet"
)

# Use dates available for both feature and target datasets
common_end = pd.Timestamp("2026-10-07")

common_dates = market_features.index.intersection(
    reference_targets.index
).sort_values()

common_dates = common_dates[common_dates <= common_end]

# Chronological 70% / 15% / 15% boundaries
n = len(common_dates)
train_cut = int(n * 0.70)
test_cut = int(n * 0.85)

# Calculate separate boundaries for each forecast horizon
split_boundaries = {}

for horizon in [5, 21]:
    split_boundaries[horizon] = {
        "Train": (
            common_dates[0],
            common_dates[train_cut - horizon - 1]
        ),
        "Train-Validation Gap": (
            common_dates[train_cut - horizon],
            common_dates[train_cut - 1]
        ),
        "Validation": (
            common_dates[train_cut],
            common_dates[test_cut - horizon - 1]
        ),
        "Validation-Test Gap": (
            common_dates[test_cut - horizon],
            common_dates[test_cut - 1]
        ),
        "Locked Test": (
            common_dates[test_cut],
            common_dates[-1]
        )
    }

for horizon, periods in split_boundaries.items():
    print(f"\n{horizon}-Day Forecast")
    for period, dates in periods.items():
        print(f"{period}: {dates[0].date()} to {dates[1].date()}")

print("\nCommon observations:", n)


5-Day Forecast
Train: 2010-01-04 to 2021-09-28
Train-Validation Gap: 2021-09-29 to 2021-10-05
Validation: 2021-10-06 to 2024-03-28
Validation-Test Gap: 2024-04-01 to 2024-04-05
Locked Test: 2024-04-08 to 2026-10-07

21-Day Forecast
Train: 2010-01-04 to 2021-09-03
Train-Validation Gap: 2021-09-06 to 2021-10-05
Validation: 2021-10-06 to 2024-03-04
Validation-Test Gap: 2024-03-05 to 2024-04-05
Locked Test: 2024-04-08 to 2026-10-07

Common observations: 4117


###  Five walk-forward folds

In [40]:
n_folds = 5
fold_results = {}

for horizon in [5, 21]:

    periods = split_boundaries[horizon]
    val_start, val_end = periods["Validation"]

    validation_dates = common_dates[
        (common_dates >= val_start) &
        (common_dates <= val_end)
    ]

    # Divide validation dates into five consecutive blocks
    fold_date_blocks = np.array_split(validation_dates, n_folds)

    folds = []

    for fold_number, fold_dates in enumerate(fold_date_blocks, start=1):

        fold_start = fold_dates[0]
        fold_end = fold_dates[-1]

        # Position of the validation start in the full date index
        start_pos = common_dates.get_loc(fold_start)

        # Exclude the forecast horizon immediately before validation
        train_end_pos = start_pos - horizon - 1

        gap_start = common_dates[train_end_pos + 1]
        gap_end = common_dates[start_pos - 1]

        folds.append({
            "Fold": fold_number,
            "Train_Start": common_dates[0],
            "Train_End": common_dates[train_end_pos],
            "Gap_Start": gap_start,
            "Gap_End": gap_end,
            "Validation_Start": fold_start,
            "Validation_End": fold_end,
            "Train_Observations": train_end_pos + 1,
            "Gap_Observations": horizon,
            "Validation_Observations": len(fold_dates)
        })

    fold_results[horizon] = pd.DataFrame(folds)

    print(f"\n{horizon}-DAY WALK-FORWARD VALIDATION")
    display(fold_results[horizon])


5-DAY WALK-FORWARD VALIDATION


,Fold,Train_Start,Train_End,Gap_Start,Gap_End,Validation_Start,Validation_End,Train_Observations,Gap_Observations,Validation_Observations
0,1,2010-01-04,2021-09-28,2021-09-29,2021-10-05,2021-10-06,2022-04-04,2876,5,123
1,2,2010-01-04,2022-03-28,2022-03-29,2022-04-04,2022-04-05,2022-09-30,2999,5,123
2,3,2010-01-04,2022-09-23,2022-09-26,2022-09-30,2022-10-03,2023-03-29,3122,5,123
3,4,2010-01-04,2023-03-22,2023-03-23,2023-03-29,2023-03-31,2023-09-27,3245,5,122
4,5,2010-01-04,2023-09-20,2023-09-21,2023-09-27,2023-09-28,2024-03-28,3367,5,122



21-DAY WALK-FORWARD VALIDATION


,Fold,Train_Start,Train_End,Gap_Start,Gap_End,Validation_Start,Validation_End,Train_Observations,Gap_Observations,Validation_Observations
0,1,2010-01-04,2021-09-03,2021-09-06,2021-10-05,2021-10-06,2022-03-30,2860,21,120
1,2,2010-01-04,2022-02-25,2022-02-28,2022-03-30,2022-03-31,2022-09-22,2980,21,120
2,3,2010-01-04,2022-08-23,2022-08-24,2022-09-22,2022-09-23,2023-03-15,3100,21,119
3,4,2010-01-04,2023-02-13,2023-02-14,2023-03-15,2023-03-16,2023-09-07,3219,21,119
4,5,2010-01-04,2023-08-08,2023-08-09,2023-09-07,2023-09-08,2024-03-04,3338,21,119
